# Third-Party Risk Intelligence — Step 4: Risk Assessment, Tiering & Actions

**UBS Student Mini-Hackathon · Scenario 2 — "Can we detect risks before they hit UBS?"**

This notebook is the **assessment layer** on top of `tpr_mistral.py`. It does not re-collect or
re-classify anything: it reads what the Mistral pipeline produced and turns it into
*severity, tiering, evidence and an owner with a deadline*.

## Where this notebook sits in the pipeline

```
tpr_mistral.py ingest     data/signals.csv + data/vendors.csv -> ingested.json
                          (public signals + UBS dependency profiles; seed_category held out)
tpr_mistral.py classify   mistral-small -> enriched_signals.jsonl                   <- WHAT kind of risk
                          category / sentiment / maturity / confidence / is_about_vendor
tpr_mistral.py warn       mistral-large agent -> warnings.jsonl (analyst-style narrative)

RISK ASSESSMENT  <== THIS NOTEBOOK
   4a. Gate       entity check, risk vs positive evidence, provenance            <- is it EVEN a signal
   4b. Score      Likelihood x Impact -> inherent risk                           <- HOW BAD
   4c. Controls   positive evidence reduces it -> residual risk
   4d. Tier       Low / Medium / High / Critical against a stated risk appetite
   4e. Route      (category x tier) -> owner team + actions + deadline + evidence
   4f. Cross-check the deterministic tiers against the LLM agent's own warnings
Dashboard / alerts        scored_signals.csv, vendor_summary.csv, alerts.json
```

**Why the split matters for the brief.** The LLM is good at *reading* a signal
(is this about the vendor, is it cyber or conduct, is it a rumour or confirmed). It is a bad
place to keep *scoring policy*: the bank has to be able to point at a number and say where it
came from. So every number below is computed in plain Python from the LLM's labels plus the
vendor's UBS dependency profile — the same discipline `tpr_mistral.py` uses when it gives the
agent a `compute_risk_score` tool instead of letting it invent scores.

## Methodology

| Concept | How it is implemented here |
|---|---|
| **Risk = Likelihood × Impact** | Both on a 1–5 scale → inherent risk 1–25 |
| **Likelihood** | Evidence strength (source credibility, signal maturity, classifier confidence, corroboration) discounted by recency — all fields the classifier already emits |
| **Impact (profiled risk)** | Category base impact, raised by the vendor's UBS dependency (`data_sensitivity`, `business_criticality`, `substitutability`) and scaled by how confirmed the UBS relationship is (`ubs_link`) |
| **Residual risk** | Inherent risk reduced by **positive** signals (certs maintained, DR test passed, remediation verified), weighted by how self-serving the source is |
| **Risk appetite** | Tier thresholds in one config cell |
| **Ethical early warning** | Unconfirmed vendor match dropped; rumours/allegations cannot auto-escalate; unconfirmed UBS links cannot reach Critical; illustrative evidence is labelled as such; every alert carries its source rows |
| **Cross-functional stakeholders** | Each category routes to an owner team; TPRM always informed |
| **Human review** | Required for every High/Critical and every capped signal — the system recommends, a person decides |

> **Data honesty.** The dataset mixes `REAL` public signals (with URLs) about real companies and
> `ILLUSTRATIVE` rows written for the demo. Nothing here is a UBS statement about a supplier;
> the `ubs_link` field records whether a UBS relationship is publicly confirmed, reported, or
> not claimed at all.

## Data contract

### INPUT 1 — `enriched_signals.jsonl` (one JSON object per line, from `tpr_mistral.py classify`)

| Field | Type | Description |
|---|---|---|
| `signal_id` | str | Unique ID |
| `vendor_id`, `vendor` | str | Vendor key and name (matched upstream via `aliases`) |
| `date` | ISO date | When the signal was published / raised |
| `source_type` | str | `regulatory_filing`, `audit_finding`, `incident_ticket`, `news_article`, `market_data`, `vendor_questionnaire` |
| `source_name`, `url` | str | Evidence pointer (URL present for `REAL` rows) |
| `text` | str | The signal itself |
| `provenance` | str | `REAL` (public source) or `ILLUSTRATIVE` (written for the demo) |
| `classification.category` | str | `CYBER`, `CONDUCT`, `COMPLIANCE`, `CONCENTRATION`, `OPERATIONAL`, `FINANCIAL`, `OTHER`, `IRRELEVANT` (dropped by the entity gate) |
| `classification.sentiment` | str | `risk` / `neutral` / `positive` |
| `classification.maturity` | str | `rumor` / `allegation` / `corroborated` / `confirmed` / `n/a` |
| `classification.confidence` | float 0–1 | Classifier confidence |
| `classification.is_about_vendor` | bool | Entity-match check (homonym guard) |
| `classification.reason` | str | One-line justification, reused in the alert |

### INPUT 2 — `ingested.json` → `profiles` (UBS dependency metadata, from `tpr_mistral.py ingest`)

| Field | Type | Description |
|---|---|---|
| `vendor_id`, `service`, `country` | str | Identity |
| `ubs_link` | str | `CONFIRMED` / `REPORTED` / `INDUSTRY` — how established the UBS relationship is |
| `ubs_link_source`, `relationship_note` | str | The public evidence for that link |
| `data_sensitivity` | 1–5 | What the vendor touches (5 = client / confidential data) |
| `business_criticality` | 1–5 | How much UBS business depends on it |
| `substitutability` | 1–5 | **1 = hard to replace** → concentration exposure |

`ingested.json` also keeps `seed_category`, the label deliberately **hidden from the
classifier**, which section 4b uses to report how trustworthy the input labels are.

### OUTPUT
1. **`scored_signals`** — risk-bearing signals plus `likelihood`, `impact`, `inherent_risk`,
   `control_reduction`, `residual_risk`, `tier`, `needs_human_review`, `explanation`.
2. **`vendor_summary`** — one row per vendor: worst residual risk, tier, dominant category,
   UBS link, dependency index, next review date.
3. **`action_items`** — one row per (vendor × category) above the alert threshold: owner team,
   response, deadline, and the evidence rows behind it. This is the early warning.
4. Files: `scored_signals.csv`, `vendor_summary.csv`, `action_items.csv`, `alerts.json`.

In [18]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

# Paths resolve next to the notebook, exactly like tpr_mistral.py resolves its own.
BASE_DIR = Path.cwd()
INGESTED = BASE_DIR / "ingested.json"              # tpr_mistral.py ingest
ENRICHED = BASE_DIR / "enriched_signals.jsonl"     # tpr_mistral.py classify
AGENT_WARNINGS = BASE_DIR / "warnings.jsonl"       # tpr_mistral.py warn (optional)

AS_OF = pd.Timestamp("2026-09-22")   # assessment date; use pd.Timestamp.today() in production

## 1. Configuration — all assumptions in one place

Everything a judge might ask "why this number?" about lives here, so the team can tune it and
explain it. These are **illustrative weights**, not UBS policy. Every key below is a value the
Mistral classifier or the vendor profile actually emits, so nothing is invented at scoring time.

In [19]:
# ---- Likelihood: how confident are we that this risk is real? -----------------
# Source credibility (0-1). A regulator filing or our own audit outranks a news story;
# a vendor answering its own questionnaire is the weakest evidence of good health.
SOURCE_CREDIBILITY = {
    "regulatory_filing":    1.00,
    "audit_finding":        0.95,
    "incident_ticket":      0.85,
    "news_article":         0.75,
    "market_data":          0.70,
    "vendor_questionnaire": 0.55,
}
# Maturity as labelled by the classifier: how far the story has hardened.
MATURITY_WEIGHT = {
    "confirmed": 1.00, "corroborated": 0.85, "allegation": 0.60,
    "rumor": 0.40, "n/a": 0.50,
}
# Likelihood = evidence strength x freshness. The weights below build evidence strength
# (0-1); recency then discounts it, because a confirmed breach from 2020 is history, not
# an early warning.
EVIDENCE_WEIGHTS = {          # must sum to 1
    "credibility":   0.35,    # trustworthy source?
    "maturity":      0.30,    # rumour or confirmed?
    "confidence":    0.15,    # is the classifier sure of the label?
    "corroboration": 0.20,    # do independent kinds of source say the same thing?
}
RECENCY_HALF_LIFE_DAYS = 365   # the dataset spans years; a signal halves every 12 months
RECENCY_FLOOR = 0.15           # an old event never drops to zero - it stays as background
CORROBORATION_WINDOW_DAYS = 365
CORROBORATION_CAP = 3          # 3+ distinct source types on the same vendor+category = fully corroborated

# ---- Impact: how much would it hurt UBS? --------------------------------------
CATEGORY_BASE_IMPACT = {       # 1-5, before the vendor profile is applied
    "CYBER":         4.5,      # breach at a supplier holding UBS data
    "COMPLIANCE":    4.0,      # regulator / auditor action
    "CONDUCT":       4.0,      # fraud, insider issues, sanctions
    "CONCENTRATION": 3.5,      # sole source, no alternative provider
    "OPERATIONAL":   3.0,      # outage, SLA breach, capacity
    "FINANCIAL":     3.0,      # vendor solvency, covenant, rating
    "OTHER":         1.5,      # contextual, no clear risk type
}
# Which profile dimension modulates which category.
DATA_SENSITIVE_CATEGORIES = {"CYBER", "COMPLIANCE", "CONDUCT"}
SUBSTITUTION_SENSITIVE_CATEGORIES = {"CONCENTRATION", "OPERATIONAL", "FINANCIAL"}
PROFILE_SPREAD = 0.5           # each dimension can move impact by +-0.5

# How established the UBS relationship is. INDUSTRY = the risk is real for the sector
# but no UBS exposure is claimed, so it must not be presented as a UBS exposure.
UBS_LINK_FACTOR = {"CONFIRMED": 1.00, "REPORTED": 0.85, "INDUSTRY": 0.65}

# ---- Residual risk: positive evidence is a control ----------------------------
# The classifier marks certs maintained / DR tests passed / remediation verified as
# sentiment="positive". Each such signal on the same vendor+category buys a reduction,
# discounted when the vendor is the one making the claim.
CONTROL_CREDIT = {
    "audit_finding":        0.10,
    "regulatory_filing":    0.10,
    "incident_ticket":      0.05,
    "news_article":         0.04,
    "market_data":          0.03,
    "vendor_questionnaire": 0.03,   # self-reported
}
CONTROL_LOOKBACK_DAYS = 365
MAX_CONTROL_REDUCTION = 0.30   # controls never remove more than 30% (residual risk is never 0)

# ---- Tiers = risk appetite (on residual risk, scale 1-25) ---------------------
TIERS = [(16, "Critical"), (10, "High"), (5, "Medium"), (0, "Low")]
ALERT_TIERS = {"High", "Critical"}         # what becomes an action item
LOW_CONFIDENCE = 0.6                       # below this -> human must verify
UNVERIFIED_MATURITY = {"rumor", "allegation"}   # cannot auto-escalate on their own
MAX_TIER_BY_UBS_LINK = {"INDUSTRY": "High"}     # unconfirmed UBS exposure -> never Critical
REVIEW_FREQUENCY_DAYS = {"Critical": 0, "High": 365, "Medium": 540, "Low": 900}

assert abs(sum(EVIDENCE_WEIGHTS.values()) - 1) < 1e-9, "evidence weights must sum to 1"

## 2. Action playbook — category decides **who**, tier decides **how fast**

Owner teams follow the cross-functional stakeholder model (Risk Management, Procurement,
Security & IT, Audit & Compliance, Data Privacy). TPRM (third-party risk management) is always
informed as the central owner of the vendor relationship. Note what is *not* here: no step
contacts a vendor, restricts access or touches a contract automatically.

In [20]:
OWNER_TEAM = {
    "CYBER":         "Security & IT (Cyber Incident Response)",
    "COMPLIANCE":    "Audit & Compliance",
    "CONDUCT":       "Compliance & Financial Crime",
    "CONCENTRATION": "Procurement & Vendor Management (Concentration Risk)",
    "OPERATIONAL":   "Business Continuity / Service Management",
    "FINANCIAL":     "Procurement & Vendor Management (Financial Health)",
    "OTHER":         "Third-Party Risk Management (triage)",
}
ALWAYS_INFORM = "Third-Party Risk Management (TPRM)"

TIER_RESPONSE = {   # deadline in hours
    "Critical": {"deadline_h": 24,   "base": "Escalate immediately; open incident; human review before any vendor contact"},
    "High":     {"deadline_h": 72,   "base": "Notify owner team; request vendor statement/attestation"},
    "Medium":   {"deadline_h": 336,  "base": "Add to watchlist; check at next monitoring cycle"},
    "Low":      {"deadline_h": None, "base": "Log only; review at scheduled reassessment"},
}

CATEGORY_ACTIONS = {   # specific steps for High/Critical
    "CYBER": [
        "Establish whether the vendor holds UBS data or has access to UBS systems/credentials",
        "Request incident details, blast radius and remediation timeline from the vendor",
        "Check indicators of compromise and exposed credentials against internal telemetry",
    ],
    "COMPLIANCE": [
        "Review contract clauses and the regulatory obligations touched by the finding",
        "Assess whether UBS regulatory reporting or notification duties are triggered",
        "Request the remediation plan / updated certification and a re-audit date",
    ],
    "CONDUCT": [
        "Refer to Financial Crime for screening of the named parties",
        "Check whether the conduct touches UBS engagements, staff or data",
        "Review the supplier code of conduct and escalation clauses in the contract",
    ],
    "CONCENTRATION": [
        "Map which UBS services depend on this vendor and with what volume",
        "Identify credible alternative suppliers and the switching cost/lead time",
        "Confirm the documented exit plan exists and has been tested",
    ],
    "OPERATIONAL": [
        "Review SLA performance and the business impact of the outage/degradation",
        "Activate or test the business continuity plan for the affected service",
        "Agree a service improvement plan with a review date",
    ],
    "FINANCIAL": [
        "Review vendor financial health, rating actions and payment exposure",
        "Check prepayments, escrow and contractual termination rights",
        "Consider enhanced monitoring or a contingency supplier",
    ],
    "OTHER": [
        "Triage: confirm the signal is in scope and re-categorise it",
    ],
}

## 3. Load the Mistral pipeline output

`tpr_mistral.py` is the collection + classification stage; this cell only reads its files.
Nothing is generated here, so the notebook fails loudly if the pipeline has not been run.

```bash
export MISTRAL_API_KEY=...
python tpr_mistral.py ingest      # -> ingested.json
python tpr_mistral.py classify    # -> enriched_signals.jsonl
python tpr_mistral.py warn        # -> warnings.jsonl (optional, used in section 11)
```

In [21]:
def load_pipeline_output(enriched_path=ENRICHED, ingested_path=INGESTED):
    # enriched_signals.jsonl + ingested.json -> (signals_df, vendors_df, truth_df)
    if not enriched_path.exists() or not ingested_path.exists():
        raise FileNotFoundError(
            f"Missing pipeline output ({enriched_path.name} / {ingested_path.name}). "
            "Run: python tpr_mistral.py ingest && python tpr_mistral.py classify")

    records = [json.loads(line) for line in enriched_path.read_text().splitlines() if line.strip()]
    rows = []
    for r in records:
        c = r["classification"]
        rows.append({
            "signal_id": r["signal_id"], "vendor_id": r["vendor_id"], "vendor_name": r["vendor"],
            "date": pd.Timestamp(r["date"]), "source_type": r["source_type"],
            "source_name": r["source_name"] or r["source_type"], "url": r.get("url", ""),
            "text": r["text"], "provenance": r.get("provenance", "ILLUSTRATIVE"),
            "risk_category": c["category"], "sentiment": c["sentiment"],
            "maturity": c.get("maturity", "n/a"),
            "classifier_confidence": float(c.get("confidence", 0.0)),
            "is_about_vendor": bool(c.get("is_about_vendor", True)),
            "classifier_reason": c.get("reason", ""),
        })
    signals = pd.DataFrame(rows)

    ing = json.loads(ingested_path.read_text())
    vendors = pd.DataFrame([{"vendor_name": name, **p} for name, p in ing["profiles"].items()])
    vendors["dependency_index"] = (                      # 0-1 summary of UBS exposure
        (vendors.data_sensitivity + vendors.business_criticality
         + (6 - vendors.substitutability)) / 15).round(3)

    # seed_category is held out of the classifier's view; kept only to grade it in 4b.
    truth = pd.DataFrame([{"signal_id": s["signal_id"], "seed_category": s["seed_category"],
                           "provenance": s["provenance"]} for s in ing["signals"]])
    return signals, vendors, truth

signals_df, vendors_df, truth_df = load_pipeline_output()
print(f"{len(signals_df)} classified signals · {len(vendors_df)} vendor profiles")
print(f"{(signals_df.provenance == 'REAL').sum()} REAL / "
      f"{(signals_df.provenance == 'ILLUSTRATIVE').sum()} ILLUSTRATIVE")
vendors_df[["vendor_id", "vendor_name", "service", "ubs_link", "data_sensitivity",
            "business_criticality", "substitutability", "dependency_index"]]

210 classified signals · 15 vendor profiles
74 REAL / 136 ILLUSTRATIVE


,vendor_id,vendor_name,service,ubs_link,data_sensitivity,business_criticality,substitutability,dependency_index
0,V001,Chain IQ Group AG,Sourcing & procurement BPO,CONFIRMED,5,4,2,0.867
1,V002,Microsoft Corporation,Cloud infrastructure & productivity,CONFIRMED,5,5,1,1.000
2,V003,LSEG,Market data & analytics,CONFIRMED,3,5,2,0.800
3,V004,Broadridge Financial Solutions,Wealth platform & post-trade,CONFIRMED,5,4,2,0.867
4,V005,Cognizant Technology Solutions,IT & business process outsourcing,CONFIRMED,4,3,4,0.600
5,V006,BlackRock,Portfolio & risk platform,REPORTED,4,4,3,0.733
6,V007,SIX Group,Digital asset issuance venue,CONFIRMED,3,3,2,0.667
7,V008,ION Group,Trading & derivatives software,INDUSTRY,4,5,1,0.933
8,V009,Capita plc,Business process outsourcing,INDUSTRY,5,3,3,0.733
9,V010,CrowdStrike Holdings,Endpoint security,INDUSTRY,5,5,2,0.933


## 4. Input validation and governance gates

Fail early if the pipeline hands over something unexpected, then apply the gates that make this
an *ethical* early-warning system rather than a rumour amplifier:

1. **Entity gate** — `is_about_vendor = false` (a homonym, a different company) is dropped, not scored.
2. **Direction gate** — only `sentiment = "risk"` signals carry risk. `positive` rows become
   control evidence in section 5; `neutral` rows stay as context.
3. **Orphan gate** — a signal for a vendor that is not in the inventory cannot be scored.

Everything dropped is counted, so the coverage is auditable.

In [22]:
REQUIRED_SIGNAL_COLS = {"signal_id", "vendor_id", "vendor_name", "date", "source_name",
                        "source_type", "url", "text", "provenance", "risk_category",
                        "sentiment", "maturity", "classifier_confidence", "is_about_vendor"}
REQUIRED_VENDOR_COLS = {"vendor_id", "vendor_name", "ubs_link", "data_sensitivity",
                        "business_criticality", "substitutability"}

def validate_inputs(signals, vendors):
    missing = REQUIRED_SIGNAL_COLS - set(signals.columns)
    assert not missing, f"signals missing fields: {missing}"
    missing = REQUIRED_VENDOR_COLS - set(vendors.columns)
    assert not missing, f"vendor profiles missing fields: {missing}"
    assert signals.signal_id.is_unique, "duplicate signal_id in enriched_signals.jsonl"
    bad_cat = set(signals.risk_category) - set(CATEGORY_BASE_IMPACT) - {"IRRELEVANT"}
    assert not bad_cat, f"unknown risk categories: {bad_cat}"
    bad_src = set(signals.source_type) - set(SOURCE_CREDIBILITY)
    assert not bad_src, f"unknown source types: {bad_src}"
    bad_mat = set(signals.maturity) - set(MATURITY_WEIGHT)
    assert not bad_mat, f"unknown maturity values: {bad_mat}"
    bad_link = set(vendors.ubs_link) - set(UBS_LINK_FACTOR)
    assert not bad_link, f"unknown ubs_link values: {bad_link}"
    orphans = set(signals.vendor_id) - set(vendors.vendor_id)
    assert not orphans, f"signals for unknown vendors: {orphans}"
    assert signals.classifier_confidence.between(0, 1).all(), "confidence must be in [0,1]"
    print(f"OK: {len(signals)} signals, {len(vendors)} vendors, schema valid")

validate_inputs(signals_df, vendors_df)

# ---- gates --------------------------------------------------------------------
wrong_entity = signals_df[~signals_df.is_about_vendor | (signals_df.risk_category == "IRRELEVANT")]
gated = signals_df[signals_df.is_about_vendor & (signals_df.risk_category != "IRRELEVANT")]

risk_signals    = gated[gated.sentiment == "risk"].reset_index(drop=True)
positive_signals = gated[gated.sentiment == "positive"].reset_index(drop=True)
neutral_signals  = gated[gated.sentiment == "neutral"].reset_index(drop=True)

print(f"dropped as wrong entity : {len(wrong_entity)}")
print(f"risk-bearing signals    : {len(risk_signals)}  -> scored")
print(f"positive evidence       : {len(positive_signals)}  -> reduces residual risk")
print(f"neutral context         : {len(neutral_signals)}  -> kept, not scored")
print()
print(pd.crosstab(gated.risk_category, gated.sentiment))

OK: 210 signals, 15 vendors, schema valid
dropped as wrong entity : 0
risk-bearing signals    : 115  -> scored
positive evidence       : 92  -> reduces residual risk
neutral context         : 3  -> kept, not scored

sentiment      neutral  positive  risk
risk_category                         
COMPLIANCE           0        39    18
CONCENTRATION        0         8    20
CONDUCT              0         0    11
CYBER                0        10    37
FINANCIAL            0         0     6
OPERATIONAL          2        35    23
OTHER                1         0     0


### 4b. How trustworthy is the input? — classifier vs held-out ground truth

The assessment can only be as good as the labels underneath it, so before scoring anything we
grade the classifier against `seed_category`, which `tpr_mistral.py` deliberately never shows to
the model (the same check as `python tpr_mistral.py validate`). This also detects the common
demo failure mode: `enriched_signals.jsonl` left over from an older run of `data/signals.csv`.

In [23]:
graded = risk_signals.merge(truth_df.drop(columns=["provenance"]), on="signal_id", how="left")
scorable = graded[graded.seed_category.notna() & ~graded.seed_category.isin(["IRRELEVANT"])]
agreement = (scorable.seed_category == scorable.risk_category).mean() if len(scorable) else float("nan")

all_ids, truth_ids = set(signals_df.signal_id), set(truth_df.signal_id)
stale, unclassified = all_ids - truth_ids, truth_ids - all_ids

print(f"classifier agreement vs held-out seed_category: {agreement:.1%} "
      f"(on {len(scorable)} gradeable risk signals)")
print(f"classified signals not in the current ingest : {len(stale)}")
print(f"ingested signals not yet classified          : {len(unclassified)}")
if stale or unclassified:
    print("  -> enriched_signals.jsonl is out of step with data/signals.csv; "
          "re-run: python tpr_mistral.py classify")

# Where does it disagree? Confusions are risk-relevant: CONCENTRATION vs OPERATIONAL is
# a routing error (different owner team), CYBER vs COMPLIANCE changes the impact weight.
if len(scorable):
    confusion = pd.crosstab(scorable.seed_category, scorable.risk_category)
    display(confusion)

classifier agreement vs held-out seed_category: 56.7% (on 97 gradeable risk signals)
classified signals not in the current ingest : 50
ingested signals not yet classified          : 110
  -> enriched_signals.jsonl is out of step with data/signals.csv; re-run: python tpr_mistral.py classify


risk_category,COMPLIANCE,CONCENTRATION,CONDUCT,CYBER,FINANCIAL,OPERATIONAL
seed_category,,,,,,
COMPLIANCE,8,3,1,2,1,0
CONDUCT,2,0,8,0,0,0
CYBER,1,2,0,25,0,7
FINANCIAL,2,0,1,0,5,0
OPERATIONAL,2,5,1,7,0,9
OTHER,0,2,0,2,0,1


## 5. Scoring functions

**Likelihood (1–5) = evidence strength × freshness.** Evidence strength is a weighted mix of
source credibility, the maturity the classifier assigned, its confidence and corroboration
(distinct *kinds* of source — audit, regulator, press — reporting the same vendor + category
within a year). Freshness is exponential decay with a floor, so a confirmed 2020 breach keeps a
small background weight instead of competing with this month's finding.

**Impact (1–5)** — how much it would hurt UBS: the category's base impact, moved by the vendor's
dependency profile (`data_sensitivity` for cyber/compliance/conduct, `substitutability` for
concentration/operational/financial, `business_criticality` always), then scaled by `ubs_link`
so a vendor with no claimed UBS relationship cannot look like a UBS exposure.

**Inherent risk = L × I** (1–25). **Residual risk** = inherent × (1 − control reduction), where
the control reduction comes from recent **positive** signals in the same vendor + category.

In [24]:
def recency_score(dates, as_of=AS_OF, half_life=RECENCY_HALF_LIFE_DAYS):
    age_days = (as_of - dates).dt.days.clip(lower=0)
    return np.power(0.5, age_days / half_life)

def freshness(dates, floor=RECENCY_FLOOR):
    # decay to a floor, not to zero: an old confirmed event is background, not noise
    return floor + (1 - floor) * recency_score(dates)

def corroboration_score(signals, window=CORROBORATION_WINDOW_DAYS,
                        cap=CORROBORATION_CAP, as_of=AS_OF):
    # Distinct KINDS of source telling the same story about the same vendor+category.
    # An audit finding plus a regulator filing plus press is corroboration; three audit
    # findings from the same review programme are one point of view.
    recent = signals[(as_of - signals["date"]).dt.days.between(0, window)]
    n_sources = (recent.groupby(["vendor_id", "risk_category"])["source_type"]
                       .nunique().rename("n_sources"))
    out = signals.join(n_sources, on=["vendor_id", "risk_category"])["n_sources"].fillna(1)
    return ((out - 1) / (cap - 1)).clip(0, 1), out.astype(int)   # 1 source -> 0, cap -> 1

def compute_likelihood(signals):
    w = EVIDENCE_WEIGHTS
    cred = signals["source_type"].map(SOURCE_CREDIBILITY)
    mat = signals["maturity"].map(MATURITY_WEIGHT)
    fresh = freshness(signals["date"])
    corr, n_src = corroboration_score(signals)
    evidence = (w["credibility"] * cred + w["maturity"] * mat
                + w["confidence"] * signals["classifier_confidence"]
                + w["corroboration"] * corr)
    return pd.DataFrame({
        "credibility": cred.round(2), "maturity_weight": mat.round(2),
        "freshness": fresh.round(3), "corroboration": corr.round(2), "n_sources": n_src,
        "evidence_strength": evidence.round(3),
        "likelihood": (1 + 4 * evidence * fresh).round(2),      # 0-1 -> 1-5
    }, index=signals.index)

def compute_impact(row):
    impact = CATEGORY_BASE_IMPACT[row.risk_category]
    # business criticality always counts; 3 is neutral, 5 adds +0.5, 1 removes -0.5
    impact += PROFILE_SPREAD * (row.business_criticality - 3) / 2
    if row.risk_category in DATA_SENSITIVE_CATEGORIES:
        impact += PROFILE_SPREAD * (row.data_sensitivity - 3) / 2
    if row.risk_category in SUBSTITUTION_SENSITIVE_CATEGORIES:
        # substitutability 1 = hard to replace = higher impact, so invert it
        impact += PROFILE_SPREAD * ((6 - row.substitutability) - 3) / 2
    impact *= UBS_LINK_FACTOR[row.ubs_link]
    return float(np.clip(impact, 1, 5))

def control_reduction_table(positives, as_of=AS_OF, lookback=CONTROL_LOOKBACK_DAYS):
    # Positive evidence (certs kept, DR test passed, remediation verified) lowers residual
    # risk for the same vendor+category. Vendor self-reporting is worth least.
    if positives.empty:
        return pd.Series(dtype=float), pd.Series(dtype=int)
    p = positives[(as_of - positives["date"]).dt.days.between(0, lookback)].copy()
    if p.empty:
        return pd.Series(dtype=float), pd.Series(dtype=int)
    p["credit"] = (p["source_type"].map(CONTROL_CREDIT).fillna(0.03)
                   * p["classifier_confidence"] * recency_score(p["date"]))
    grouped = p.groupby(["vendor_id", "risk_category"])
    return (grouped["credit"].sum().clip(upper=MAX_CONTROL_REDUCTION),
            grouped.size().rename("n_positive"))

def assign_tier(score):
    for lower, name in TIERS:
        if score >= lower:
            return name

TIER_ORDER = ["Low", "Medium", "High", "Critical"]

def cap_tier(tier, max_tier):
    # never raise a tier, only hold it down
    return tier if TIER_ORDER.index(tier) <= TIER_ORDER.index(max_tier) else max_tier

## 6. Explanations — every score must say *why*

The brief asks for "a risk level / score + short explanation of why it was flagged" and the
"source/evidence behind the warning". The explanation is assembled from the same fields the
score used, so it can never drift from the number, and it ends with the classifier's own
one-line reason.

In [25]:
def explain(row):
    parts = [
        f"{row.tier.upper()} ({row.residual_risk:.1f}/25, L{row.likelihood:.1f} x I{row.impact:.1f})",
        f"{row.risk_category} signal, {row.maturity}, from {row.source_type.replace('_', ' ')} "
        f"(credibility {row.credibility:.2f})",
        f"{row.n_sources} independent kind(s) of source in {CORROBORATION_WINDOW_DAYS}d",
        f"{(AS_OF - row.date).days}d old (freshness {row.freshness:.2f})",
        f"UBS link: {row.ubs_link.lower()} ({row.relationship_note.rstrip('.')})",
    ]
    if row.risk_category in DATA_SENSITIVE_CATEGORIES:
        parts.append(f"data sensitivity {row.data_sensitivity}/5")
    if row.risk_category in SUBSTITUTION_SENSITIVE_CATEGORIES and row.substitutability <= 2:
        parts.append(f"hard to replace (substitutability {row.substitutability}/5)")
    if row.control_reduction > 0:
        parts.append(f"{row.n_positive} positive signal(s) cut risk by {row.control_reduction:.0%}")
    if row.classifier_confidence < LOW_CONFIDENCE:
        parts.append(f"LOW classifier confidence ({row.classifier_confidence:.2f}) - verify manually")
    if row.maturity_cap:
        parts.append(f"{row.maturity} only - capped below High until a human confirms it")
    if row.link_cap:
        parts.append(f"UBS exposure not confirmed - capped at {MAX_TIER_BY_UBS_LINK[row.ubs_link]}")
    if row.provenance != "REAL":
        parts.append("ILLUSTRATIVE evidence (demo data, not a public source)")
    parts.append(f"classifier: {row.classifier_reason}")
    return "; ".join(parts)

## 7. Run the assessment

In [26]:
VENDOR_COLS = ["vendor_id", "service", "country", "ubs_link", "ubs_link_source",
               "relationship_note", "data_sensitivity", "business_criticality",
               "substitutability", "dependency_index"]

def assess_signals(risk, positives, vendors):
    df = risk.merge(vendors[VENDOR_COLS], on="vendor_id", how="left")
    df = pd.concat([df, compute_likelihood(df)], axis=1)
    df["impact"] = df.apply(compute_impact, axis=1).round(2)
    df["inherent_risk"] = (df["likelihood"] * df["impact"]).round(2)

    red, n_pos = control_reduction_table(positives)
    key = list(zip(df.vendor_id, df.risk_category))
    df["control_reduction"] = [round(float(red.get(k, 0.0)), 3) for k in key]
    df["n_positive"] = [int(n_pos.get(k, 0)) for k in key]
    df["residual_risk"] = (df["inherent_risk"] * (1 - df["control_reduction"])).round(2)

    raw_tier = df["residual_risk"].apply(assign_tier)

    # Governance cap 1: a rumour or an allegation never auto-escalates on its own. It is held
    # just below the High threshold until a human confirms it.
    high_floor = dict((n, l) for l, n in TIERS)["High"]
    df["maturity_cap"] = (df["maturity"].isin(UNVERIFIED_MATURITY)
                          & (df["residual_risk"] >= high_floor))
    df.loc[df["maturity_cap"], "residual_risk"] = high_floor - 0.01

    # Governance cap 2: where the UBS relationship is not confirmed, the finding may be real
    # for the industry but must not be raised as a UBS exposure at the top tier.
    tier = df["residual_risk"].apply(assign_tier)
    max_tier = df["ubs_link"].map(MAX_TIER_BY_UBS_LINK)
    capped = [cap_tier(t, m) if isinstance(m, str) else t for t, m in zip(tier, max_tier)]
    df["link_cap"] = [c != t for c, t in zip(capped, tier)]
    df["tier"] = capped

    df["needs_human_review"] = (df["tier"].isin(ALERT_TIERS)
                                | (df["classifier_confidence"] < LOW_CONFIDENCE)
                                | df["maturity_cap"]
                                | (df["provenance"] != "REAL") & df["tier"].isin(ALERT_TIERS))
    df["tier_before_caps"] = raw_tier
    df["explanation"] = df.apply(explain, axis=1)
    return df.sort_values("residual_risk", ascending=False).reset_index(drop=True)

scored_signals = assess_signals(risk_signals, positive_signals, vendors_df)
print(f"{len(scored_signals)} risk signals scored · "
      + str(scored_signals.tier.value_counts()
            .reindex(TIER_ORDER[::-1], fill_value=0).to_dict()))
scored_signals[["signal_id", "vendor_name", "risk_category", "maturity", "source_type",
                "likelihood", "impact", "inherent_risk", "control_reduction",
                "residual_risk", "tier", "needs_human_review"]].head(15)

115 risk signals scored · {'Critical': 1, 'High': 30, 'Medium': 53, 'Low': 31}


,signal_id,vendor_name,risk_category,maturity,source_type,likelihood,impact,inherent_risk,control_reduction,residual_risk,tier,needs_human_review
0,I0140,SIX Group,CYBER,confirmed,audit_finding,3.85,4.50,17.32,0.000,17.32,Critical,True
1,I0200,Infosys,CYBER,corroborated,incident_ticket,3.89,4.04,15.72,0.000,15.72,High,True
2,I0102,LSEG,CONCENTRATION,confirmed,audit_finding,3.64,4.25,15.47,0.000,15.47,High,True
3,I0086,Microsoft Corporation,CONCENTRATION,confirmed,audit_finding,3.37,4.50,15.16,0.000,15.16,High,True
4,I0089,Microsoft Corporation,OPERATIONAL,confirmed,incident_ticket,3.97,4.00,15.88,0.104,14.23,High,True
5,I0083,Microsoft Corporation,OPERATIONAL,confirmed,incident_ticket,3.85,4.00,15.40,0.104,13.80,High,True
6,I0130,BlackRock,CYBER,confirmed,audit_finding,3.08,4.25,13.09,0.000,13.09,High,True
7,I0193,Infosys,CYBER,confirmed,audit_finding,3.22,4.04,13.01,0.000,13.01,High,True
8,I0082,Microsoft Corporation,OPERATIONAL,corroborated,audit_finding,3.52,4.00,14.08,0.104,12.62,High,True
9,R0036,Chain IQ Group AG,CYBER,confirmed,incident_ticket,2.49,5.00,12.45,0.000,12.45,High,True


In [27]:
# The demo outputs the brief asks for, in words: what, how bad, why, and on what evidence.
for _, r in scored_signals.head(4).iterrows():
    print(f"[{r.signal_id}] {r.vendor_name} ({r.ubs_link}) | {r.risk_category}")
    print(f"   signal   : {r.text}")
    print(f"   source   : {r.source_name} ({r.source_type}, {r.date.date()}, {r.provenance})"
          + (f" {r.url}" if r.url else ""))
    print(f"   why      : {r.explanation}")
    print()

[I0140] SIX Group (CONFIRMED) | CYBER
   signal   : Access recertification for SIX Group was overdue, with dormant privileged accounts still active on the engagement.
   source   : audit_finding (audit_finding, 2026-07-27, ILLUSTRATIVE)
   why      : CRITICAL (17.3/25, L3.9 x I4.5); CYBER signal, confirmed, from audit finding (credibility 0.95); 1 independent kind(s) of source in 365d; 57d old (freshness 0.91); UBS link: confirmed (UBS among issuers on the SIX Digital Exchange platform); data sensitivity 3/5; ILLUSTRATIVE evidence (demo data, not a public source); classifier: Overdue access recertification and active dormant privileged accounts indicate a cybersecurity risk exposure.

[I0200] Infosys (REPORTED) | CYBER
   signal   : Phishing simulation across the Infosys engagement showed click rates above the agreed threshold.
   source   : incident_ticket (incident_ticket, 2026-07-20, ILLUSTRATIVE)
   why      : HIGH (15.7/25, L3.9 x I4.0); CYBER signal, corroborated, from incident t

## 8. Vendor-level summary

A vendor's risk in a category = the **highest** residual risk among its signals in that category
(a single serious breach must not be averaged away by routine good news). The vendor's overall
risk = its worst category. Vendors with no risk signals at all still appear, at 0 — an empty row
is itself information for the dashboard.

In [28]:
def summarise_vendors(scored, vendors):
    cat = (scored.groupby(["vendor_id", "risk_category"])
                 .agg(residual_risk=("residual_risk", "max"),
                      n_signals=("signal_id", "count"),
                      n_real=("provenance", lambda s: int((s == "REAL").sum())))
                 .reset_index())
    worst = cat.loc[cat.groupby("vendor_id")["residual_risk"].idxmax()] if len(cat) else cat
    summary = (vendors[["vendor_id", "vendor_name", "service", "ubs_link", "dependency_index"]]
               .merge(worst[["vendor_id", "risk_category", "residual_risk"]]
                      .rename(columns={"risk_category": "dominant_category"}),
                      on="vendor_id", how="left"))
    summary["n_risk_signals"] = summary.vendor_id.map(scored.groupby("vendor_id").size()).fillna(0).astype(int)
    summary["n_positive"] = summary.vendor_id.map(
        positive_signals.groupby("vendor_id").size()).fillna(0).astype(int)
    summary["residual_risk"] = summary["residual_risk"].fillna(0)
    summary["tier"] = summary["residual_risk"].apply(assign_tier)
    summary["next_review"] = summary["tier"].map(
        lambda t: (AS_OF + pd.Timedelta(days=REVIEW_FREQUENCY_DAYS[t])).date())
    return summary.sort_values("residual_risk", ascending=False).reset_index(drop=True), cat

vendor_summary, vendor_category_risk = summarise_vendors(scored_signals, vendors_df)
vendor_summary

,vendor_id,vendor_name,service,ubs_link,dependency_index,dominant_category,residual_risk,n_risk_signals,n_positive,tier,next_review
0,V007,SIX Group,Digital asset issuance venue,CONFIRMED,0.667,CYBER,17.32,6,7,Critical,2026-09-22
1,V012,Infosys,IT services & BPO,REPORTED,0.600,CYBER,15.72,7,7,High,2027-09-22
2,V003,LSEG,Market data & analytics,CONFIRMED,0.800,CONCENTRATION,15.47,6,8,High,2027-09-22
3,V002,Microsoft Corporation,Cloud infrastructure & productivity,CONFIRMED,1.000,CONCENTRATION,15.16,6,8,High,2027-09-22
4,V006,BlackRock,Portfolio & risk platform,REPORTED,0.733,CYBER,13.09,5,9,High,2027-09-22
5,V001,Chain IQ Group AG,Sourcing & procurement BPO,CONFIRMED,0.867,CYBER,12.45,9,4,High,2027-09-22
6,V004,Broadridge Financial Solutions,Wealth platform & post-trade,CONFIRMED,0.867,CONCENTRATION,12.40,4,10,High,2027-09-22
7,V005,Cognizant Technology Solutions,IT & business process outsourcing,CONFIRMED,0.600,CYBER,11.63,6,7,High,2027-09-22
8,V008,ION Group,Trading & derivatives software,INDUSTRY,0.933,COMPLIANCE,11.00,10,4,High,2027-09-22
9,V010,CrowdStrike Holdings,Endpoint security,INDUSTRY,0.933,COMPLIANCE,10.71,11,3,High,2027-09-22


## 9. Threshold → action items (the early warnings)

Only (vendor × category) pairs in **High** or **Critical** become action items. Medium stays on
the watchlist, Low is logged. Each item carries the owner team, the deadline, the concrete steps
and the evidence rows the tier was built from — including whether that evidence is a real public
source or demo data.

In [29]:
EVIDENCE_LIMIT = 5

def build_action_items(scored, vendor_cat, vendors):
    if vendor_cat.empty:
        return pd.DataFrame()
    alerts = vendor_cat.copy()
    alerts["tier"] = alerts["residual_risk"].apply(assign_tier)
    alerts = alerts[alerts["tier"].isin(ALERT_TIERS)]
    profile = vendors.set_index("vendor_id")
    rows = []
    for _, a in alerts.iterrows():
        ev = (scored[(scored.vendor_id == a.vendor_id) & (scored.risk_category == a.risk_category)]
              .sort_values("residual_risk", ascending=False))
        top = ev.head(EVIDENCE_LIMIT)
        p = profile.loc[a.vendor_id]
        resp = TIER_RESPONSE[a.tier]
        rows.append({
            "vendor_id": a.vendor_id,
            "vendor_name": p.vendor_name,
            "ubs_link": p.ubs_link,
            "risk_category": a.risk_category,
            "tier": a.tier,
            "residual_risk": round(float(a.residual_risk), 2),
            "owner_team": OWNER_TEAM[a.risk_category],
            "inform": ALWAYS_INFORM,
            "deadline": (AS_OF + pd.Timedelta(hours=resp["deadline_h"])).strftime("%Y-%m-%d %H:%M"),
            "response": resp["base"],
            "actions": CATEGORY_ACTIONS[a.risk_category],
            "human_review_required": True,
            "n_signals": int(a.n_signals),
            "n_real_sources": int(a.n_real),
            "evidence_basis": "public sources" if a.n_real else "ILLUSTRATIVE demo data only",
            "evidence": [{"signal_id": s.signal_id, "date": str(s.date.date()),
                          "source": s.source_name, "source_type": s.source_type,
                          "provenance": s.provenance, "maturity": s.maturity,
                          "text": s.text, "url": s.url}
                         for s in top.itertuples()],
            "why": top.explanation.iloc[0],
        })
    return (pd.DataFrame(rows).sort_values(["residual_risk"], ascending=False)
            .reset_index(drop=True))

action_items = build_action_items(scored_signals, vendor_category_risk, vendors_df)
print(f"{len(action_items)} early warnings "
      f"({(action_items.tier == 'Critical').sum() if len(action_items) else 0} Critical, "
      f"{(action_items.tier == 'High').sum() if len(action_items) else 0} High)")
action_items[["vendor_name", "ubs_link", "risk_category", "tier", "residual_risk",
              "owner_team", "deadline", "evidence_basis"]]

18 early warnings (1 Critical, 17 High)


,vendor_name,ubs_link,risk_category,tier,residual_risk,owner_team,deadline,evidence_basis
0,SIX Group,CONFIRMED,CYBER,Critical,17.32,Security & IT (Cyber Incident Response),2026-09-23 00:00,ILLUSTRATIVE demo data only
1,Infosys,REPORTED,CYBER,High,15.72,Security & IT (Cyber Incident Response),2026-09-25 00:00,public sources
2,LSEG,CONFIRMED,CONCENTRATION,High,15.47,Procurement & Vendor Management (Concentration Risk),2026-09-25 00:00,public sources
3,Microsoft Corporation,CONFIRMED,CONCENTRATION,High,15.16,Procurement & Vendor Management (Concentration Risk),2026-09-25 00:00,public sources
4,Microsoft Corporation,CONFIRMED,OPERATIONAL,High,14.23,Business Continuity / Service Management,2026-09-25 00:00,ILLUSTRATIVE demo data only
5,BlackRock,REPORTED,CYBER,High,13.09,Security & IT (Cyber Incident Response),2026-09-25 00:00,ILLUSTRATIVE demo data only
6,Chain IQ Group AG,CONFIRMED,CYBER,High,12.45,Security & IT (Cyber Incident Response),2026-09-25 00:00,public sources
7,Broadridge Financial Solutions,CONFIRMED,CONCENTRATION,High,12.40,Procurement & Vendor Management (Concentration Risk),2026-09-25 00:00,ILLUSTRATIVE demo data only
8,Chain IQ Group AG,CONFIRMED,COMPLIANCE,High,12.33,Audit & Compliance,2026-09-25 00:00,ILLUSTRATIVE demo data only
9,LSEG,CONFIRMED,COMPLIANCE,High,11.80,Audit & Compliance,2026-09-25 00:00,ILLUSTRATIVE demo data only


In [30]:
# Full alert cards - this is what the dashboard renders and what the pitch reads out.
for _, a in action_items.head(6).iterrows():
    print("=" * 100)
    print(f"{a.tier.upper()}: {a.vendor_name} | {a.risk_category} | {a.residual_risk:.1f}/25 "
          f"| UBS link: {a.ubs_link}")
    print(f"Owner: {a.owner_team}   (inform: {a.inform})   Deadline: {a.deadline}")
    print(f"Response: {a.response}")
    print(f"Why flagged: {a.why}")
    for step in a.actions:
        print(f"  - {step}")
    print(f"Evidence ({a.n_signals} signal(s), {a.n_real_sources} from public sources):")
    for e in a.evidence:
        tail = f" | {e['url']}" if e["url"] else ""
        print(f"  * [{e['signal_id']}] {e['date']} | {e['source']} ({e['source_type']}, "
              f"{e['maturity']}, {e['provenance']}){tail}")
        print(f"      \"{e['text']}\"")
    print()

CRITICAL: SIX Group | CYBER | 17.3/25 | UBS link: CONFIRMED
Owner: Security & IT (Cyber Incident Response)   (inform: Third-Party Risk Management (TPRM))   Deadline: 2026-09-23 00:00
Response: Escalate immediately; open incident; human review before any vendor contact
Why flagged: CRITICAL (17.3/25, L3.9 x I4.5); CYBER signal, confirmed, from audit finding (credibility 0.95); 1 independent kind(s) of source in 365d; 57d old (freshness 0.91); UBS link: confirmed (UBS among issuers on the SIX Digital Exchange platform); data sensitivity 3/5; ILLUSTRATIVE evidence (demo data, not a public source); classifier: Overdue access recertification and active dormant privileged accounts indicate a cybersecurity risk exposure.
  - Establish whether the vendor holds UBS data or has access to UBS systems/credentials
  - Request incident details, blast radius and remediation timeline from the vendor
  - Check indicators of compromise and exposed credentials against internal telemetry
Evidence (1 signa

## 10. Visual check — risk heat map (Likelihood × Impact)

The classic risk matrix; each dot is a scored signal, coloured by how confirmed the UBS
relationship is. Useful slide for the pitch: the top-right quadrant is the early-warning zone,
and the pale dots there are the ones the governance caps hold back.

In [31]:
import plotly.graph_objects as go

grid = np.arange(1, 5.01, 0.05)
z = np.outer(grid, grid)   # impact (rows) x likelihood (cols)
fig = go.Figure(go.Heatmap(
    x=grid, y=grid, z=z, showscale=False, hoverinfo="skip",
    colorscale=[[0, "#2e7d32"], [5/25, "#fbc02d"], [10/25, "#ef6c00"], [16/25, "#c62828"], [1, "#7f0000"]],
    opacity=0.45))

MARKERS = {"CONFIRMED": ("#111111", "circle"), "REPORTED": ("#37474f", "diamond"),
           "INDUSTRY": ("#90a4ae", "x")}
for link, (colour, symbol) in MARKERS.items():
    part = scored_signals[scored_signals.ubs_link == link]
    if part.empty:
        continue
    fig.add_trace(go.Scatter(
        x=part.likelihood, y=part.impact, mode="markers", name=f"UBS link: {link.lower()}",
        marker=dict(size=9, color=colour, symbol=symbol, line=dict(width=0.5, color="white")),
        hovertext=(part.vendor_name + " | " + part.risk_category + " | " + part.tier
                   + "<br>" + part.text.str.slice(0, 90)), hoverinfo="text"))

fig.update_layout(title="Vendor risk signals on the Likelihood x Impact matrix (inherent risk)",
                  xaxis_title="Likelihood (1-5)", yaxis_title="Impact (1-5)",
                  width=820, height=620, legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.7)"))
fig.show()

ModuleNotFoundError: No module named 'plotly'

## 11. Cross-check — deterministic tiers vs the Mistral agent's own warnings

`tpr_mistral.py warn` runs a `mistral-large` agent over the same signals with tools
(`query_signals`, `get_vendor_profile`, `compute_risk_score`, `raise_warning`) and produces its
own narrative warnings. Two independent routes to the same conclusion is a much stronger pitch
than either alone — and a disagreement is a finding, not a bug: it shows where a judgement call
sits. This section is skipped if `warnings.jsonl` has not been produced.

In [ ]:
if AGENT_WARNINGS.exists():
    agent = pd.DataFrame([json.loads(l) for l in AGENT_WARNINGS.read_text().splitlines() if l.strip()])
    agent = agent.rename(columns={"vendor": "vendor_name", "level": "agent_level"})
    agent["agent_level"] = agent["agent_level"].str.title()
    mine = (vendor_summary[["vendor_name", "tier", "residual_risk"]]
            .rename(columns={"tier": "notebook_tier"}))
    cmp = mine.merge(agent[["vendor_name", "agent_level", "reason"]], on="vendor_name", how="left")
    cmp["agent_level"] = cmp["agent_level"].fillna("(no warning)")
    cmp["agrees"] = cmp.notebook_tier == cmp.agent_level
    print(f"agent warnings: {len(agent)} · tier agreement on "
          f"{cmp.agrees.sum()}/{len(cmp)} vendors")
    display(cmp)
else:
    print(f"{AGENT_WARNINGS.name} not found - run `python tpr_mistral.py warn` to enable "
          "the cross-check. The deterministic assessment above does not depend on it.")

warnings.jsonl not found - run `python tpr_mistral.py warn` to enable the cross-check. The deterministic assessment above does not depend on it.


## 12. Export outputs for the dashboard

In [ ]:
scored_signals.to_csv(BASE_DIR / "scored_signals.csv", index=False)
vendor_summary.to_csv(BASE_DIR / "vendor_summary.csv", index=False)
if len(action_items):
    action_items.drop(columns=["actions", "evidence"]).to_csv(BASE_DIR / "action_items.csv", index=False)
    with open(BASE_DIR / "alerts.json", "w") as f:
        json.dump(action_items.to_dict(orient="records"), f, indent=2, default=str)
print("Wrote scored_signals.csv, vendor_summary.csv, action_items.csv, alerts.json")

Wrote scored_signals.csv, vendor_summary.csv, action_items.csv, alerts.json


## 13. Sanity tests (normal + difficult cases)

Assertions over the live pipeline output, not over hand-picked rows, so they keep working after
`tpr_mistral.py classify` is re-run on new signals. They document what the model is *expected*
to do, especially in the cases that are easy to get wrong.

In [ ]:
# --- arithmetic must hold ------------------------------------------------------
assert scored_signals.likelihood.between(1, 5).all(), "likelihood out of range"
assert scored_signals.impact.between(1, 5).all(), "impact out of range"
assert (scored_signals.residual_risk > 0).all(), "residual risk is never zero"
assert (scored_signals.residual_risk <= scored_signals.inherent_risk + 1e-9).all()
assert (scored_signals.control_reduction <= MAX_CONTROL_REDUCTION + 1e-9).all()

# --- the gates ----------------------------------------------------------------
# Only risk-bearing, correctly-matched signals are ever scored.
assert scored_signals.is_about_vendor.all(), "a wrong-entity signal was scored"
assert (scored_signals.sentiment == "risk").all(), "a positive/neutral signal was scored"
# A rumour or allegation cannot auto-escalate into an alert on its own.
unverified = scored_signals[scored_signals.maturity.isin(UNVERIFIED_MATURITY)]
assert not unverified.tier.isin(ALERT_TIERS).any(), "unverified signal escalated without a human"
assert unverified.needs_human_review.all() if len(unverified) else True
# A vendor with no confirmed UBS relationship never reaches Critical.
industry = scored_signals[scored_signals.ubs_link == "INDUSTRY"]
assert not (industry.tier == "Critical").any(), "unconfirmed UBS exposure raised to Critical"
# Every alert is reviewed by a human and carries its evidence.
if len(action_items):
    assert action_items.human_review_required.all()
    assert action_items.evidence.map(len).gt(0).all(), "an alert has no evidence"
    known_ids = set(scored_signals.signal_id)
    assert all(e["signal_id"] in known_ids for ev in action_items.evidence for e in ev), \
        "an alert cites a signal_id that is not in the scored set"

# --- risk-appetite behaviour ---------------------------------------------------
# Higher UBS dependency must not lower the score for the same category and evidence.
probe = pd.DataFrame([{**scored_signals.iloc[0].to_dict(), "business_criticality": bc}
                      for bc in (1, 5)])
impacts = probe.apply(compute_impact, axis=1)
assert impacts.iloc[1] >= impacts.iloc[0], "criticality must not reduce impact"
# Positive evidence must reduce, never increase, residual risk.
with_controls = scored_signals[scored_signals.control_reduction > 0]
assert (with_controls.residual_risk < with_controls.inherent_risk).all() if len(with_controls) else True
# Every scored signal is explainable.
assert scored_signals.explanation.str.len().gt(40).all(), "an explanation is missing/too short"

print(f"All sanity tests passed on {len(scored_signals)} scored signals "
      f"and {len(action_items)} alerts")

All sanity tests passed on 115 scored signals and 18 alerts


## Limitations & next steps (for the pitch)

- **The scores are only as good as the labels.** Section 4b grades the classifier against a
  held-out `seed_category`; every point of disagreement there is a mis-routed alert here
  (`CONCENTRATION` vs `OPERATIONAL` sends the case to a different team). Next step: fold the
  confusion matrix back into the classifier prompt, and re-grade on every run.
- **Weights are expert assumptions, not learned.** They are all in section 1 for exactly that
  reason. Calibrate with analyst feedback (thumbs up/down on alerts) or fit them once enough
  labelled outcomes exist.
- **Entity matching happens upstream.** `is_about_vendor` is the homonym guard; errors there
  propagate, so low-confidence signals always go to human review rather than into an alert.
- **Self-reported evidence is weighted down, not trusted.** A vendor questionnaire can lower a
  score by at most a few points and never below 70% of inherent risk.
- **Mixed provenance.** `REAL` rows come from public sources and carry URLs; `ILLUSTRATIVE` rows
  were written for the demo. Alerts state which they rest on, and an alert built only on
  illustrative evidence is marked as such rather than quietly presented as fact.
- **Public signals only, and only about companies.** No personal data, no paywalled or scraped
  private sources, no inference about named individuals. `ubs_link` distinguishes a confirmed
  UBS supplier from a company that simply serves the industry — an INDUSTRY-linked finding is
  never presented as a UBS exposure and is capped below Critical.
- **No automatic decisions.** The system recommends an owner, a deadline and a set of steps; a
  person decides on vendor contact, access restriction or contract action.